In [1]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi not found (no GPU runtime?)")
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(), "| CUDA build:", torch.version.cuda)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    gb = p.total_memory / 1024**3
    print(f"GPU: {p.name} | {gb:.1f} GB | compute capability {p.major}.{p.minor}")
else:
    print("WARNING: no GPU. Switch the runtime type to GPU.")

Thu Nov 26 02:22:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["STUDIO_DATA_DIR"] = "/content/drive/MyDrive/TextToImageStudio"
else:
    os.environ["STUDIO_DATA_DIR"] = "/content/studio_data"
os.makedirs(os.environ["STUDIO_DATA_DIR"], exist_ok=True)
print("Data directory:", os.environ["STUDIO_DATA_DIR"])

Mounted at /content/drive
Data directory: /content/drive/MyDrive/TextToImageStudio


In [3]:
import os, shutil, zipfile, subprocess
TARGET = "/content/Text-to-Image-Studio-AI"
shutil.rmtree(TARGET, ignore_errors=True)          # remove any old copy so the fixed code is used

PROJECT_ZIP = "/content/drive/MyDrive/Text-to-Image-Studio-AI.zip"
REPO_URL = ""
if REPO_URL:
    subprocess.run(["git", "clone", REPO_URL, TARGET], check=True)
elif os.path.exists(PROJECT_ZIP):
    with zipfile.ZipFile(PROJECT_ZIP) as z: z.extractall("/content")
    print("Extracted ZIP to /content")
else:
    raise FileNotFoundError("Upload Text-to-Image-Studio-AI.zip to Drive (or set REPO_URL), then re-run this cell.")
os.chdir(TARGET)
print(os.listdir("."))

Extracted ZIP to /content
['notebooks', 'modules', 'TESTING.md', 'requirements.txt', 'services', 'data', 'app.py', '.gitignore', 'components', 'assets', 'tests', 'tools', 'config', '.streamlit', 'models', 'views', 'README.md']


In [4]:
!pip install -q -r requirements.txt

In [5]:
DOWNLOAD_SD15, DOWNLOAD_SDXL, DOWNLOAD_CLIP = True, False, True
from huggingface_hub import snapshot_download
if DOWNLOAD_SD15:
    snapshot_download("stable-diffusion-v1-5/stable-diffusion-v1-5", allow_patterns=["*.json","*.txt","*fp16.safetensors","tokenizer/*","scheduler/*"])
if DOWNLOAD_SDXL:
    snapshot_download("stabilityai/stable-diffusion-xl-base-1.0", allow_patterns=["*.json","*.txt","*fp16.safetensors","tokenizer*/*","scheduler/*"])
if DOWNLOAD_CLIP:
    snapshot_download("openai/clip-vit-base-patch32")
print("Model cache ready.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Model cache ready.


In [6]:
!python -m unittest discover -s tests 2>&1 | tail -n 15

............................................./content/Text-to-Image-Studio-AI/tests/test_torch_modules.py:19: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  self.assertLessEqual(float(x.abs().max()), 1.0)
...
----------------------------------------------------------------------
Ran 48 tests in 22.123s

OK


In [7]:
import subprocess, time, re, os, urllib.request, stat
os.chdir("/content/Text-to-Image-Studio-AI")
subprocess.run(["pkill", "-f", "streamlit run"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)

log = open("streamlit.log", "w")
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
                  "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"], stdout=log, stderr=subprocess.STDOUT)
time.sleep(6)

CF = "/content/cloudflared"
if not os.path.exists(CF):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
    os.chmod(CF, os.stat(CF).st_mode | stat.S_IEXEC)
tunnel = subprocess.Popen([CF, "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"], stderr=subprocess.PIPE, text=True)
url = None
t0 = time.time()
while time.time() - t0 < 60 and url is None:
    line = tunnel.stderr.readline()
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line or "")
    if m: url = m.group(0)
print("OPEN THIS LINK:", url or "no link found, see the troubleshooting cell below")

OPEN THIS LINK: https://attacks-harry-lotus-parental.trycloudflare.com


In [8]:
!tail -n 30 /content/Text-to-Image-Studio-AI/streamlit.log
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv


  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.218.94:8501

memory.used [MiB], memory.total [MiB]
3 MiB, 15360 MiB


In [9]:
import os, shutil
p = "/content/Text-to-Image-Studio-AI/views/generator.py"
t = open(p).read()
old = 's.gen_model = r[0].selectbox("Model"'
new = 'r[0].selectbox("Model"'
print("old line found:", old in t)
open(p, "w").write(t.replace(old, new))

# clear cached bytecode so Python uses the patched file
shutil.rmtree("/content/Text-to-Image-Studio-AI/views/__pycache__", ignore_errors=True)

# show the result: the line must now start with r[0].selectbox, not s.gen_model =
for i, line in enumerate(open(p), 1):
    if 'selectbox("Model"' in line:
        print(i, line.strip()[:90])

old line found: False
343 st.selectbox("Model", list(models), key="gen_model", format_func=lambda k: f"{k} ({'ready'


In [10]:
!tail -n 40 /content/Text-to-Image-Studio-AI/streamlit.log


  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.218.94:8501



In [11]:
import os, shutil, zipfile
from google.colab import drive
drive.mount("/content/drive")
os.environ["STUDIO_DATA_DIR"] = "/content/drive/MyDrive/TextToImageStudio"
os.makedirs(os.environ["STUDIO_DATA_DIR"], exist_ok=True)

TARGET = "/content/Text-to-Image-Studio-AI"
shutil.rmtree(TARGET, ignore_errors=True)
with zipfile.ZipFile("/content/drive/MyDrive/Text-to-Image-Studio-AI.zip") as z:
    z.extractall("/content")
os.chdir(TARGET)
print(os.listdir("."))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['notebooks', 'modules', 'TESTING.md', 'requirements.txt', 'services', 'data', 'app.py', '.gitignore', 'components', 'assets', 'tests', 'tools', 'config', '.streamlit', 'models', 'views', 'README.md']


In [12]:
!pip install -q -r /content/Text-to-Image-Studio-AI/requirements.txt

In [13]:
g = open("/content/Text-to-Image-Studio-AI/views/generator.py").read()
m = open("/content/Text-to-Image-Studio-AI/services/model_manager.py").read()
e = open("/content/Text-to-Image-Studio-AI/modules/text_embeddings.py").read()
print("home fix:", "s.gen_model = r[0].selectbox" not in g)
print("vae fix:", "pipe.enable_vae_slicing()" not in m)
print("embedding fix:", "pooler_output" in e)

home fix: True
vae fix: False
embedding fix: True


In [14]:
import subprocess, time, re, os, urllib.request, stat
os.chdir("/content/Text-to-Image-Studio-AI")
os.environ["STUDIO_DATA_DIR"] = "/content/drive/MyDrive/TextToImageStudio"
subprocess.run(["pkill", "-f", "streamlit run"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
log = open("streamlit.log", "w")
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
                  "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"], stdout=log, stderr=subprocess.STDOUT)
time.sleep(6)
CF = "/content/cloudflared"
if not os.path.exists(CF):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
    os.chmod(CF, os.stat(CF).st_mode | stat.S_IEXEC)
tunnel = subprocess.Popen([CF, "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"], stderr=subprocess.PIPE, text=True)
url, t0 = None, time.time()
while time.time() - t0 < 60 and url is None:
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", tunnel.stderr.readline() or "")
    if m: url = m.group(0)
print("OPEN THIS LINK:", url or "no link found")

OPEN THIS LINK: https://leaving-included-acts-sailing.trycloudflare.com


In [15]:
m = open("/content/Text-to-Image-Studio-AI/services/model_manager.py").read()
print("vae fix:", "pipe.vae.enable_slicing()" in m)

vae fix: True


In [16]:
ROOT = "/content/Text-to-Image-Studio-AI"
p = f"{ROOT}/views/generator.py"
s = open(p).read()
old = '''                mx = cfg["max_side"]
                cc[0].slider("Width", 256, mx, step=8, key="gen_w")'''
new = '''                mx = cfg["max_side"]
                # A reused/upscaled image can be larger than this model allows (e.g. 2048 > 1536): keep the stored size inside the slider range.
                s.gen_w = max(256, min(int(s.gen_w), mx)) // 8 * 8
                s.gen_h = max(256, min(int(s.gen_h), mx)) // 8 * 8
                cc[0].slider("Width", 256, mx, step=8, key="gen_w")'''
if "s.gen_w = max(256" in s:
    print("[already fixed]")
elif old in s:
    open(p, "w").write(s.replace(old, new)); print("[patched] generator.py")
else:
    print("[NOT FOUND] send me a screenshot of this cell's output")
import shutil; shutil.rmtree(f"{ROOT}/views/__pycache__", ignore_errors=True)

[patched] generator.py


In [17]:
from pathlib import Path

root = Path("/content/Text-to-Image-Studio-AI")

# 1) views/lora.py: give the caption table a key the "remember my inputs" loop never touches
f = root / "views" / "lora.py"; s = f.read_text()
old = 'num_rows="fixed", key="lora_caption_editor",'
new = 'num_rows="fixed", key=f"capedit_{s.lora_dataset}",   # not a persisted prefix; one editor state per dataset\n                            '
if 'key=f"capedit_' in s:
    print("lora.py already patched")
elif old in s:
    f.write_text(s.replace(old, new, 1)); print("lora.py patched")
else:
    print("lora.py: pattern not found")

# 2) app.py: skip data-editor state in the persistence loop (safety net)
a = root / "app.py"; t = a.read_text()
if "NEVER_PERSIST" in t:
    print("app.py already patched")
else:
    edits = [
        ('PERSISTENT_BOOLS = {"gen_hq", "gen_upscale", "gen_use_lora", "gen_save", "gen_enh_model", "ds_pp_flip", "ds_sp_use", "gal_fav", "ev_show_all"}\n',
         'PERSISTENT_BOOLS = {"gen_hq", "gen_upscale", "gen_use_lora", "gen_save", "gen_enh_model", "ds_pp_flip", "ds_sp_use", "gal_fav", "ev_show_all"}\n'
         '# st.data_editor keys hold an edit log that Streamlit forbids assigning (StreamlitValueAssignmentNotAllowedError), so they are never re-assigned.\n'
         'NEVER_PERSIST = {"lora_caption_editor"}\n'),
        ('        if key.startswith(PERSISTENT_PREFIXES):\n            value = st.session_state[key]\n',
         '        if key.startswith(PERSISTENT_PREFIXES) and key not in NEVER_PERSIST:\n            value = st.session_state[key]\n'
         '            if isinstance(value, dict) and "edited_rows" in value:   # any other data_editor state\n                continue\n'),
    ]
    missing = [x[:50] for x, _ in edits if x not in t]
    if missing:
        print("app.py: pattern not found:", missing)
    else:
        for x, y in edits:
            t = t.replace(x, y, 1)
        a.write_text(t); print("app.py patched")
print("Done. Restart the Streamlit cell, then open the NEW link in a fresh tab.")

lora.py patched
app.py patched
Done. Restart the Streamlit cell, then open the NEW link in a fresh tab.


In [18]:
from pathlib import Path

p = Path("/content/Text-to-Image-Studio-AI/modules/dataset_explorer.py")
s = p.read_text()

if "is_caption" in s:
    print("Already patched.")
else:
    edits = [
        ('''        if Path(name).suffix.lower() not in IMAGE_EXTS:
            continue
        target = (dest / name).resolve()''',
         '''        is_image = Path(name).suffix.lower() in IMAGE_EXTS
        # captions travel with the images: a captions.csv and small .txt caption files are kept too
        is_caption = (Path(name).name.lower() == "captions.csv" and info.file_size <= 50 * 1024 ** 2) or (Path(name).suffix.lower() == ".txt" and info.file_size <= 20_000)
        if not (is_image or is_caption):
            continue
        target = (dest / name).resolve()'''),
        ('''            shutil.copyfileobj(src, out)
        count += 1
    if count == 0:''',
         '''            shutil.copyfileobj(src, out)
        count += 1 if is_image else 0
    if count == 0:'''),
    ]
    missing = [a[:45] for a, _ in edits if a not in s]
    if missing:
        print("Pattern not found:", missing)
    else:
        for a, b in edits:
            s = s.replace(a, b, 1)
        p.write_text(s)
        print("Patched. Restart the Streamlit cell and open the new link.")

Patched. Restart the Streamlit cell and open the new link.


In [19]:
from pathlib import Path

root = Path("/content/Text-to-Image-Studio-AI")

# ---- A) modules/dataset_explorer.py: ZIP extraction keeps captions.csv / .txt captions ----
p = root / "modules" / "dataset_explorer.py"; s = p.read_text()
if "is_caption" in s:
    print("A) dataset_explorer.py already patched")
else:
    edits = [
        ('''        if Path(name).suffix.lower() not in IMAGE_EXTS:
            continue
        target = (dest / name).resolve()''',
         '''        is_image = Path(name).suffix.lower() in IMAGE_EXTS
        # captions travel with the images: a captions.csv and small .txt caption files are kept too
        is_caption = (Path(name).name.lower() == "captions.csv" and info.file_size <= 50 * 1024 ** 2) or (Path(name).suffix.lower() == ".txt" and info.file_size <= 20_000)
        if not (is_image or is_caption):
            continue
        target = (dest / name).resolve()'''),
        ('''            shutil.copyfileobj(src, out)
        count += 1
    if count == 0:''',
         '''            shutil.copyfileobj(src, out)
        count += 1 if is_image else 0
    if count == 0:'''),
    ]
    missing = [a[:45] for a, _ in edits if a not in s]
    if missing:
        print("A) pattern not found:", missing)
    else:
        for a, b in edits:
            s = s.replace(a, b, 1)
        p.write_text(s); print("A) dataset_explorer.py patched")

# ---- B) views/lora.py: the upload box now also accepts a ZIP (images + captions.csv) and caption files ----
f = root / "views" / "lora.py"; t = f.read_text()
if "def_zip_upload_ok" in t:
    print("B) lora.py upload already patched")
else:
    old_up = 'type=["png", "jpg", "jpeg", "webp"], accept_multiple_files=True, key="up_lora_imgs")'
    new_up = 'type=["png", "jpg", "jpeg", "webp", "zip", "csv", "txt"], accept_multiple_files=True, key="up_lora_imgs",\n                              help="Images, or one ZIP containing images + captions.csv (columns: filename, caption).")  # def_zip_upload_ok'
    old_save = '            n = dx.save_uploaded_files([(f.name, f.getvalue()) for f in up], dest)\n'
    new_save = '''            import shutil
            shutil.rmtree(dest, ignore_errors=True)            # re-saving a name replaces the old set instead of mixing images
            dest.mkdir(parents=True, exist_ok=True)
            n = 0
            for f in up:
                ext = Path(f.name).suffix.lower()
                if ext == ".zip":
                    try:
                        n += dx.safe_extract_zip(f.getvalue(), dest)
                    except Exception as exc:
                        notify(f"Could not read '{f.name}': {exc}", "error")
                elif ext in dx.IMAGE_EXTS:
                    (dest / Path(f.name).name).write_bytes(f.getvalue()); n += 1
                elif f.name.lower() == "captions.csv" or ext == ".txt":
                    (dest / Path(f.name).name).write_bytes(f.getvalue())
            dx.flatten_single_root(dest)
            if n == 0:
                notify("No images found in the upload.", "error")
                st.stop()
'''
    missing = [x[:45] for x in (old_up, old_save) if x not in t]
    if missing:
        print("B) pattern not found:", missing)
    else:
        t = t.replace(old_up, new_up, 1).replace(old_save, new_save, 1)
        f.write_text(t); print("B) lora.py upload patched")

# ---- C) views/lora.py + app.py: the caption table must not be re-assigned by the "remember inputs" loop ----
t = f.read_text()
old = 'num_rows="fixed", key="lora_caption_editor",'
new = 'num_rows="fixed", key=f"capedit_{s.lora_dataset}",   # not a persisted prefix; one editor state per dataset\n                            '
if 'key=f"capedit_' in t:
    print("C) lora.py caption table already patched")
elif old in t:
    f.write_text(t.replace(old, new, 1)); print("C) lora.py caption table patched")
else:
    print("C) lora.py: caption table pattern not found")

a = root / "app.py"; u = a.read_text()
if "NEVER_PERSIST" in u:
    print("C) app.py already patched")
else:
    edits = [
        ('PERSISTENT_BOOLS = {"gen_hq", "gen_upscale", "gen_use_lora", "gen_save", "gen_enh_model", "ds_pp_flip", "ds_sp_use", "gal_fav", "ev_show_all"}\n',
         'PERSISTENT_BOOLS = {"gen_hq", "gen_upscale", "gen_use_lora", "gen_save", "gen_enh_model", "ds_pp_flip", "ds_sp_use", "gal_fav", "ev_show_all"}\n'
         '# st.data_editor keys hold an edit log that Streamlit forbids assigning (StreamlitValueAssignmentNotAllowedError), so they are never re-assigned.\n'
         'NEVER_PERSIST = {"lora_caption_editor"}\n'),
        ('        if key.startswith(PERSISTENT_PREFIXES):\n            value = st.session_state[key]\n',
         '        if key.startswith(PERSISTENT_PREFIXES) and key not in NEVER_PERSIST:\n            value = st.session_state[key]\n'
         '            if isinstance(value, dict) and "edited_rows" in value:   # any other data_editor state\n                continue\n'),
    ]
    missing = [x[:45] for x, _ in edits if x not in u]
    if missing:
        print("C) app.py: pattern not found:", missing)
    else:
        for x, y in edits:
            u = u.replace(x, y, 1)
        a.write_text(u); print("C) app.py patched")
print("Done. Restart the Streamlit cell, then open the NEW link in a fresh tab.")

A) dataset_explorer.py already patched
B) lora.py upload patched
C) lora.py caption table already patched
C) app.py already patched
Done. Restart the Streamlit cell, then open the NEW link in a fresh tab.


In [20]:
from pathlib import Path

a = Path("/content/Text-to-Image-Studio-AI/app.py")
s = a.read_text()

if "_hide_outdated_torchao" in s:
    print("Already patched.")
else:
    old = "import importlib\nimport traceback\n"
    new = '''import importlib
import sys
import traceback


def _hide_outdated_torchao(min_version: str = "0.16.0") -> None:
    """Colab ships an old optional 'torchao' that newer peft/diffusers reject with an ImportError when a LoRA
    adapter is attached. This project never uses torchao, so an outdated copy is hidden from them."""
    try:
        from importlib import metadata
        from packaging.version import Version
        if "torchao" not in sys.modules and Version(metadata.version("torchao")) < Version(min_version):
            sys.modules["torchao"] = None
    except Exception:
        pass


_hide_outdated_torchao()
'''
    if old not in s:
        print("Pattern not found: app.py differs from what I expected.")
    else:
        a.write_text(s.replace(old, new, 1))
        print("Patched. Restart the Streamlit cell and open the new link.")

Patched. Restart the Streamlit cell and open the new link.


In [21]:
from pathlib import Path

root = Path("/content/Text-to-Image-Studio-AI")

# ---------- modules/image_generator.py ----------
f = root / "modules" / "image_generator.py"; s = f.read_text()

PREV_CONST = '\nSCHEDULER_FALLBACKS = ("DPM++ 2M", "Euler", "DDIM")   # tried in this order if the chosen scheduler crashes with an IndexError\n'
PREV_BLOCK = '''        used_scheduler = req.scheduler
        try:
            with torch.inference_mode():
                images = pipe(**kwargs).images
        except IndexError as first_error:
            # Some diffusers versions fail inside multistep/Karras schedulers ("index N is out of bounds for dimension 0 with size N").
            # Retry the SAME seed with simpler schedulers so the run still completes, and say so.
            images = None
            for alt in [n for n in SCHEDULER_FALLBACKS if n != req.scheduler]:
                mm.set_scheduler(pipe, alt)
                kwargs["generator"] = [torch.Generator("cpu").manual_seed(sd) for sd in seeds]
                try:
                    with torch.inference_mode():
                        images = pipe(**kwargs).images
                    used_scheduler = alt
                    warnings.append(f"Scheduler '{req.scheduler}' failed with this diffusers version ({first_error}); used '{alt}' instead.")
                    break
                except IndexError:
                    continue
            if images is None:
                raise
'''
ORIG_BLOCK = '''        with torch.inference_mode():
            images = pipe(**kwargs).images
'''

if "_BROKEN_SCHEDULERS" in s:
    print("image_generator.py already patched")
else:
    if "SCHEDULER_FALLBACKS" in s:                       # upgrade from the earlier (simpler) patch: undo it first
        s = s.replace(PREV_CONST, "", 1).replace(PREV_BLOCK, ORIG_BLOCK, 1)
        s = s.replace("scheduler=used_scheduler,\n                    steps", "scheduler=req.scheduler,\n                    steps", 1)
    NEW_CONST = '''
SCHEDULER_FALLBACKS = ("DPM++ 2M", "Euler", "DDIM")   # tried in this order if the chosen scheduler crashes with an IndexError
_BROKEN_SCHEDULERS: set = set()                         # schedulers that crashed here but had a working replacement: skipped from then on
'''
    NEW_SET = '''        chosen = req.scheduler
        if chosen in _BROKEN_SCHEDULERS:
            chosen = next((n for n in SCHEDULER_FALLBACKS if n not in _BROKEN_SCHEDULERS), "DDIM")
            warnings.append(f"Scheduler '{req.scheduler}' is known to fail with this diffusers version; using '{chosen}'.")
        mm.set_scheduler(pipe, chosen)
'''
    NEW_BLOCK = '''        used_scheduler = chosen
        try:
            with torch.inference_mode():
                images = pipe(**kwargs).images
        except IndexError as first_error:
            # Some diffusers versions fail inside multistep/Karras schedulers ("index N is out of bounds for dimension 0 with size N").
            # Retry the SAME seed with simpler schedulers so the run still completes, and say so.
            images, failed = None, [chosen]
            for alt in [n for n in SCHEDULER_FALLBACKS if n != chosen and n not in _BROKEN_SCHEDULERS]:
                mm.set_scheduler(pipe, alt)
                kwargs["generator"] = [torch.Generator("cpu").manual_seed(sd) for sd in seeds]
                try:
                    with torch.inference_mode():
                        images = pipe(**kwargs).images
                    used_scheduler = alt
                    _BROKEN_SCHEDULERS.update(failed)
                    warnings.append(f"Scheduler '{chosen}' failed with this diffusers version ({first_error}); used '{alt}' instead.")
                    break
                except IndexError:
                    failed.append(alt)
            if images is None:
                raise
'''
    edits = [
        ("from services.model_manager import get_model_manager\n", "from services.model_manager import get_model_manager\n" + NEW_CONST),
        ("        mm.set_scheduler(pipe, req.scheduler)\n", NEW_SET),
        (ORIG_BLOCK, NEW_BLOCK),
        ("model=req.model_key, scheduler=req.scheduler,\n                    steps=req.steps", "model=req.model_key, scheduler=used_scheduler,\n                    steps=req.steps"),
    ]
    missing = [e[0][:45] for e in edits if e[0] not in s]
    if missing:
        print("image_generator.py: pattern not found:", missing)
    else:
        for a, b in edits:
            s = s.replace(a, b, 1)
        f.write_text(s); print("image_generator.py patched")

# ---------- modules/evaluation.py: report the scheduler that was really used ----------
e = root / "modules" / "evaluation.py"; t = e.read_text()
old = '"scheduler": scheduler, "seed": seed, "time_s"'
new = '"scheduler": (res.records[0].get("scheduler") or scheduler) if res.records else scheduler, "seed": seed, "time_s"'
if new in t:
    print("evaluation.py already patched")
elif old in t:
    e.write_text(t.replace(old, new, 1)); print("evaluation.py patched")
else:
    print("evaluation.py: pattern not found")
print("Done. Restart the Streamlit cell and open the new link.")

image_generator.py patched
evaluation.py patched
Done. Restart the Streamlit cell and open the new link.


In [22]:
from pathlib import Path

root = Path("/content/Text-to-Image-Studio-AI")

# ================= 1) Training dashboard: values fully visible =================
g = root / "views" / "gan.py"; s = g.read_text()
if 'tiles([("Status"' in s:
    print("1) gan.py already patched")
else:
    a = s.index('        c = st.columns(5)\n        c[0].metric("Status", j.status)')
    b = s.index('        job_status_line(j)\n        if j.extra.get("params_g")')
    new = '''        run_dir = Path(j.extra.get("run_dir", ""))
        tiles([("Status", str(j.status).capitalize()),
               ("Epoch", f"{last.get('epoch', 0)}/{j.extra.get('epochs', '?')}"),
               ("Generator loss", f"{last['loss_g']:.3f}" if last else "-"),
               ("Discriminator loss", f"{last['loss_d']:.3f}" if last else "-"),
               ("Checkpoint", "saved" if (run_dir / "latest.pt").exists() else "none")])
'''
    s = s[:a] + new + s[b:]
    s = s.replace("from components.ui_components import go, job_status_line, notify, panel_title, style_fig, to_bytes",
                  "from components.ui_components import go, job_status_line, notify, panel_title, style_fig, tiles, to_bytes", 1)
    g.write_text(s); print("1) gan.py patched")

c = root / "assets" / "styles" / "theme.css"; t = c.read_text()
if "labels stay on ONE line" in t:
    print("1) theme.css already patched")
else:
    anchor = ".tile .v.long{font-size:clamp(.75rem,15cqw,1.3rem);overflow:hidden}\n"
    if anchor not in t:
        print("1) theme.css: anchor not found")
    else:
        t = t.replace(anchor, anchor + """/* labels stay on ONE line and scale down with the tile instead of wrapping or being cut off */
.tile .l{white-space:nowrap;overflow:visible;font-size:clamp(.6rem,8.5cqw,.78rem)}
div[data-testid="stMetric"]{container-type:inline-size;min-width:0}
div[data-testid="stMetricLabel"],div[data-testid="stMetricLabel"] *{white-space:nowrap!important;overflow:visible!important;text-overflow:clip!important;font-size:clamp(.62rem,8.5cqw,.85rem)!important}
div[data-testid="stMetricValue"],div[data-testid="stMetricValue"] *{white-space:nowrap!important;overflow:visible!important;text-overflow:clip!important;font-size:clamp(.8rem,13cqw,1.6rem)!important}
""", 1)
        c.write_text(t); print("1) theme.css patched")

# ================= 2) Variation: switch txt2img <-> img2img without reloading the model =================
m = root / "services" / "model_manager.py"; u = m.read_text()
if "share the weights already on the GPU" in u:
    print("2) model_manager.py already patched")
else:
    old = '''            if self._pipe is not None and self._cache_key == key:
                return self._pipe
            self.unload()
'''
    new = '''            if self._pipe is not None and self._cache_key == key:
                return self._pipe
            if self._pipe is not None and self._cache_key and self._cache_key[0] == model_key and not getattr(self, "offloaded", False):
                # Same model, other mode (e.g. Variation = img2img): share the weights already on the GPU instead of
                # unloading and re-reading several GB from disk, which is what made Variation take minutes.
                try:
                    cls = AutoPipelineForImage2Image if mode == "img2img" else AutoPipelineForText2Image
                    pipe = cls.from_pipe(self._pipe)
                    pipe.set_progress_bar_config(disable=True)
                    self._pipe, self._cache_key = pipe, key
                    return pipe
                except Exception:
                    pass                                   # fall back to a normal reload below
            self.unload()
'''
    if old not in u:
        print("2) model_manager.py: pattern not found")
    else:
        m.write_text(u.replace(old, new, 1)); print("2) model_manager.py patched")

# ================= 3) Reuse settings: visible confirmation + scroll to the filled-in form =================
v = root / "views" / "generator.py"; w = v.read_text()
if "gen_reuse_notice" in w:
    print("3) generator.py already patched")
else:
    old1 = '    s.gen_seed = int(rec["seed"]) if keep_seed and rec.get("seed") is not None else -1\n'
    new1 = old1 + '''    if s.pop("gen_reuse_notice", False):
        st.toast(f"Settings restored from this image: {rec.get('model')}, {rec['width']}×{rec['height']}, {s.gen_steps} steps, "
                 f"guidance {s.gen_guidance}, {s.gen_scheduler}, seed {s.gen_seed}. The form above is filled in.", icon="✅")
        notify("Settings restored from the selected image. Press Generate to reproduce it.", "success")
        import streamlit.components.v1 as components      # bring the filled-in form into view (best effort)
        components.html("<script>const m=window.parent.document.querySelector('[data-testid=\\"stMain\\"]')||window.parent.document.querySelector('section.main');"
                        "if(m){m.scrollTo({top:0,behavior:'smooth'});}</script>", height=0)
'''
    old2 = '                _request_run(rec, "", keep_seed=True)\n'
    new2 = '                st.session_state["gen_reuse_notice"] = True\n                _request_run(rec, "", keep_seed=True)\n'
    missing = [x[:50] for x in (old1, old2) if x not in w]
    if missing:
        print("3) generator.py: pattern not found:", missing)
    else:
        v.write_text(w.replace(old1, new1, 1).replace(old2, new2, 1)); print("3) generator.py patched")

print("Done. Restart the Streamlit cell and open the NEW link in a fresh tab.")

1) gan.py patched
1) theme.css patched
2) model_manager.py patched
3) generator.py patched
Done. Restart the Streamlit cell and open the NEW link in a fresh tab.


In [23]:
!python -m unittest discover -s tests 2>&1 | tail -n 15

............................................./content/Text-to-Image-Studio-AI/tests/test_torch_modules.py:19: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  self.assertLessEqual(float(x.abs().max()), 1.0)
...
----------------------------------------------------------------------
Ran 48 tests in 12.547s

OK
